# UniProt-Rhea mapping

This is a notebook of SPARQL-queries used in UniProt to obtain different mappings. Downloading all .rdf-files would be a drastic move, as this is a couple of hundreds of GBs, and the code would take a couple of days to run. Hence, more fruitful to simply submit the query online to UniProt, and download obtained CSV.

In [1]:
# Mapping for UniProt-Rhea (UID-RID)
query1="""
    PREFIX up: <http://purl.uniprot.org/core/>
    SELECT ?reaction ?protein
    WHERE {
       ?protein rdf:type up:Protein .
       ?protein up:annotation ?b .
       ?b up:catalyticActivity ?a .
       ?a up:catalyzedReaction ?reaction .    
    }
"""

# Mapping for UniProt-RefSeq (UID-RSID)
query2="""
    PREFIX up: <http://purl.uniprot.org/core/>
    SELECT ?protein ?refseq
    WHERE {
       ?protein rdf:type up:Protein .
       ?protein rdfs:seeAlso ?refseq .
       FILTER regex(str(?refseq), "http://purl.uniprot.org/refseq")
    }
"""

Here comes the processing of the obtained csv's from these queries.

In [2]:
import pandas as pd

def extract_id(value):
    if pd.notna(value) and isinstance(value, str):
        return value.strip().split("/")[-1] 
    return ""

def process_csv(input_file, output_file, columns):
    df = pd.read_csv(f"Raw_queries/{input_file}")
    df = df.map(extract_id)
    df.columns = columns
    df.to_csv(f"Modified_queries/{output_file}", sep="\t", index=False)

In [ ]:
process_csv("query1.csv", "RID_UID.tsv", ["RID", "UID"])
process_csv("query2.csv", "UID_RSID.tsv", ["UID", "RSID"])

Now over to a test-section, where an attempt will be done to create a giant .tsv mapping all RIDs, RSIDs and UIDs together.

In [ ]:
uid_rsid_df = pd.read_csv("Modified_queries/UID_RSID.tsv", sep="\t", dtype=str)
rid_uid_df = pd.read_csv("Modified_queries/RID_UID.tsv", sep="\t", dtype=str)
merged_df = uid_rsid_df.merge(rid_uid_df, on="UID", how="left")
merged_df.to_csv("Modified_queries/UID_RSID_RID.tsv", sep="\t", index=False)